# Binance USD-M Perpetual Full-Market Klines

This notebook shows how to use the local `binance_klines_data_fetch` package from Jupyter, load the current Binance USD-M perpetual futures universe from `/fapi/v1/exchangeInfo`, and fetch closed 1-minute klines for the full market.

Run the cleanup cell at the end when you are done because `MultiSymbolKlineService` starts a background thread.

## 1. Add The Project To `sys.path`

In [1]:
import sys
from pathlib import Path

repo_path = Path("/home/suncong/binance_klines_data_fetch")
repo_path_str = str(repo_path)
if repo_path_str not in sys.path:
    sys.path.insert(0, repo_path_str)

import pandas as pd
from IPython.display import display
from binance_klines_data_fetch import (
    MultiSymbolKlineService,
    get_um_perpetual_symbol_info,
    get_um_perpetual_symbols,
)

print("Imported package from:", repo_path)

Imported package from: /home/suncong/binance_klines_data_fetch


## 2. Load Current USD-M Perpetual Symbol Universe

This cell accesses the real Binance REST API once through `/fapi/v1/exchangeInfo`. The helper keeps only symbols where `contractType == "PERPETUAL"` and `status == "TRADING"`.

In [2]:
# Set quote_assets=None for all USD-M perpetual contracts.
# Use quote_assets=["USDT"] if you only want USDT-quoted contracts.
quote_assets = None

symbol_info = get_um_perpetual_symbol_info(quote_assets=quote_assets)
symbols = symbol_info["symbol"].tolist()

print("USD-M perpetual symbols:", len(symbols))
display(symbol_info[["symbol", "baseAsset", "quoteAsset", "marginAsset", "status"]].head(20))
print(symbols[:20])

USD-M perpetual symbols: 568


,symbol,baseAsset,quoteAsset,marginAsset,status
0,0GUSDT,0G,USDT,USDT,TRADING
1,1000000BOBUSDT,1000000BOB,USDT,USDT,TRADING
2,1000000MOGUSDT,1000000MOG,USDT,USDT,TRADING
3,1000BONKUSDC,1000BONK,USDC,USDC,TRADING
4,1000BONKUSDT,1000BONK,USDT,USDT,TRADING
5,1000CATUSDT,1000CAT,USDT,USDT,TRADING
6,1000CHEEMSUSDT,1000CHEEMS,USDT,USDT,TRADING
7,1000FLOKIUSDT,1000FLOKI,USDT,USDT,TRADING
8,1000LUNCUSDT,1000LUNC,USDT,USDT,TRADING
9,1000PEPEUSDC,1000PEPE,USDC,USDC,TRADING


['0GUSDT', '1000000BOBUSDT', '1000000MOGUSDT', '1000BONKUSDC', '1000BONKUSDT', '1000CATUSDT', '1000CHEEMSUSDT', '1000FLOKIUSDT', '1000LUNCUSDT', '1000PEPEUSDC', '1000PEPEUSDT', '1000RATSUSDT', '1000SATSUSDT', '1000SHIBUSDC', '1000SHIBUSDT', '1000XECUSDT', '1INCHUSDT', '1MBABYDOGEUSDT', '2ZUSDT', '4USDT']


## 3. Start Full-Market Background Kline Service

`window_size=20` keeps the initial full-market bootstrap modest. Increase it if you need a deeper rolling history. All symbols share one process-local weighted rate limiter.

In [6]:
service = MultiSymbolKlineService(
    symbols=symbols,
    window_size=20,
    max_workers=8,
    refresh_interval_seconds=5.0,
    startup_timeout_seconds=300.0,
)

service.start(block_until_ready=True, timeout=300.0)
print("service ready:", service.status().ready)
print("symbols loaded in service:", len(service.status().symbols))

service ready: True
symbols loaded in service: 568


## 4. Read One Symbol

In [7]:
sample_symbol = "BTCUSDT" if "BTCUSDT" in symbols else symbols[0]
sample_df = service.get_recent(sample_symbol, 10)

print(sample_symbol, "rows:", len(sample_df), "last_open_time:", sample_df.index[-1])
display(sample_df.tail())

BTCUSDT rows: 10 last_open_time: 2026-05-30 07:56:00+00:00


,Open,High,Low,Close,Volume,Close_Time,Quote_Asset_Volume,Number_of_Trades,Taker_Buy_Base_Asset_Volume,Taker_Buy_Quote_Asset_Volume
Open_Time,,,,,,,,,,
2026-05-30 07:52:00+00:00,73492.6,73493.0,73492.5,73493.0,21.996,2026-05-30 07:52:59.999000+00:00,1.616545e+06,323,13.778,1.012582e+06
2026-05-30 07:53:00+00:00,73493.0,73502.9,73493.0,73493.8,38.888,2026-05-30 07:53:59.999000+00:00,2.858203e+06,620,18.941,1.392122e+06
2026-05-30 07:54:00+00:00,73493.8,73510.3,73491.5,73510.2,35.319,2026-05-30 07:54:59.999000+00:00,2.595734e+06,638,24.823,1.824339e+06
2026-05-30 07:55:00+00:00,73510.3,73510.3,73486.2,73489.4,21.248,2026-05-30 07:55:59.999000+00:00,1.561593e+06,505,5.455,4.009261e+05
2026-05-30 07:56:00+00:00,73489.3,73504.9,73489.3,73502.0,15.290,2026-05-30 07:56:59.999000+00:00,1.123805e+06,514,8.097,5.950987e+05


## 5. Build A Full-Market Latest-Candle Summary

In [8]:
all_latest = service.get_all_recent(1)

rows = []
for symbol, df in all_latest.items():
    if df.empty:
        rows.append({"symbol": symbol, "ready": False})
        continue
    latest = df.iloc[-1]
    rows.append(
        {
            "symbol": symbol,
            "ready": True,
            "last_open_time": df.index[-1],
            "Open": float(latest["Open"]),
            "High": float(latest["High"]),
            "Low": float(latest["Low"]),
            "Close": float(latest["Close"]),
            "Volume": float(latest["Volume"]),
        }
    )

latest_summary = pd.DataFrame(rows).sort_values("symbol").reset_index(drop=True)
print("ready symbols:", int(latest_summary["ready"].sum()), "/", len(latest_summary))
display(latest_summary.head(30))

ready symbols: 568 / 568


,symbol,ready,last_open_time,Open,High,Low,Close,Volume
0,0GUSDT,True,2026-05-30 07:56:00+00:00,0.430500,0.430800,0.430500,0.430800,525.0
1,1000000BOBUSDT,True,2026-05-30 07:56:00+00:00,0.014600,0.014610,0.014590,0.014610,28937.0
2,1000000MOGUSDT,True,2026-05-30 07:56:00+00:00,0.132500,0.132500,0.132500,0.132500,3842.1
3,1000BONKUSDC,True,2026-05-30 07:56:00+00:00,0.005490,0.005493,0.005490,0.005493,85149.0
4,1000BONKUSDT,True,2026-05-30 07:56:00+00:00,0.005495,0.005499,0.005494,0.005498,368155.0
5,1000CATUSDT,True,2026-05-30 07:56:00+00:00,0.001886,0.001896,0.001885,0.001886,7554504.0
6,1000CHEEMSUSDT,True,2026-05-30 07:56:00+00:00,0.000623,0.000623,0.000623,0.000623,519223.0
7,1000FLOKIUSDT,True,2026-05-30 07:56:00+00:00,0.027980,0.028020,0.027980,0.028020,238454.0
8,1000LUNCUSDT,True,2026-05-30 07:56:00+00:00,0.080980,0.081040,0.080910,0.080970,52727.0
9,1000PEPEUSDC,True,2026-05-30 07:56:00+00:00,0.003416,0.003418,0.003416,0.003417,1650392.0


## 6. Inspect Service And Rate-Limit Status

In [ ]:
status = service.status()

print("running:", status.running)
print("ready:", status.ready)
print("last_refresh_at:", status.last_refresh_at)
print("rate_limiter:", status.rate_limiter)

symbol_status = pd.DataFrame(
    [
        {
            "symbol": symbol,
            "ready": item.ready,
            "rows": item.row_count,
            "last_open_time": item.last_open_time,
            "error": item.last_error,
        }
        for symbol, item in status.symbols.items()
    ]
).sort_values("symbol")
display(symbol_status.head(30))
display(symbol_status[symbol_status["error"].notna()].head(20))

## 7. Stop Background Thread

Run this before closing the notebook or before creating another service instance.

In [ ]:
service.stop()
print("service running:", service.status().running)